# 28 — OCR Basics
**Goal:** Extract text from scanned resume images using OCR.

Some resumes never contain text at all: they are scans, faxes, or screenshots of a printed page. Optical Character Recognition (OCR) is the branch of the pipeline that turns pixels back into characters so the rest of the NLP stack can process them like any other document.

**Why it matters for resumes / ATS:** older CVs, internationally mailed applications, and image-only exports all arrive as scans. If the pipeline has no OCR fallback, those resumes silently disappear from indexing. OCR is the safety net that keeps extraction complete — and, as this chapter shows, the noisiest input the normalizer (Ch. 29) has to clean.

## 1. When to Use OCR

OCR is only needed when there is no text layer: scanned PDFs (an image wrapped in PDF syntax) and image-only files (PNG/JPG screenshots). If `extract_text()` returns empty or gibberish, that is the signal to switch branches. The cell also surveys the tool landscape — Tesseract (open-source, fast), EasyOCR (deep-learning, more accurate), and cloud Document AI (most accurate, but a service call).

**What the code does:** prints the two trigger cases, then the tool list with the two trade-off lines: accuracy `EasyOCR > Tesseract`, speed `Tesseract > EasyOCR`.

**Expected output:** the two trigger cases, the three tool names, and the accuracy/speed comparison. Takeaway: pick the cheapest tool that clears your accuracy bar — clean scans usually pass with Tesseract; handwriting or noisy scans justify the heavier engines.

In [ ]:
print('''OCR is needed for:\n1. Scanned PDFs (image, not text)\n2. Image-only files (PNG, JPG screenshots)\n\nTools:\n- Tesseract (open-source, fast)\n- EasyOCR (deep learning, more accurate)\n- Google/AWS Document AI (cloud, most accurate)\n\nAccuracy: EasyOCR > Tesseract\nSpeed: Tesseract > EasyOCR''')

## 2. OCR with Tesseract

The cell synthesizes the whole scenario: it *draws* a fake resume with PIL (name, title, skills), saves it as PNG, and runs `pytesseract.image_to_string()` over it — the same call you would make on a real scan. Note the defensive structure: `ImportError` catches missing Python packages, and a bare `except` catches the missing Tesseract *system binary*, so the cell degrades to a printed hint instead of crashing the notebook.

**What the code does:**
- Creates a 400×200 white RGB image, draws three text lines with an Arial TTF (falling back to `ImageFont.load_default()` if the font is absent), saves to `/tmp/test_resume.png`
- OCRs the image and prints `=== OCR Result ===` plus the recognized text

**Expected:** on a machine without the Tesseract binary this prints `OCR error: tesseract is not installed or it's not in your PATH (needs Tesseract system install)` — the graceful-degradation path. With Tesseract installed, `image_to_string` returns the three drawn lines, typically with minor spacing/character artifacts; OCR output is never as clean as a native text layer.

In [ ]:
try:
    import pytesseract
    from PIL import Image, ImageDraw, ImageFont
    
    img = Image.new('RGB', (400, 200), color='white')
    draw = ImageDraw.Draw(img)
    try: font = ImageFont.truetype("arial.ttf", 24)
    except: font = ImageFont.load_default()
    draw.text((20, 20), "Srivatsa Gorti", fill='black', font=font)
    draw.text((20, 60), "Senior Data Scientist", fill='black', font=font)
    draw.text((20, 100), "Python, NLP, TensorFlow", fill='black', font=font)
    img.save("/tmp/test_resume.png")
    
    text = pytesseract.image_to_string(img)
    print("=== OCR Result ===")
    print(text)
except ImportError:
    print("Install: pip install pytesseract pillow")
except Exception as e:
    print(f"OCR error: {e} (needs Tesseract system install)")

## 3. Image Preprocessing for Better OCR

OCR accuracy is decided before OCR runs: Tesseract performs best on clean, high-contrast, binarized input. The standard chain is grayscale → contrast enhancement → thresholding, turning a gray-on-white scan into black-on-white pixels.

**What the code does:**
- Opens the PNG from the previous cell and converts it to grayscale (`"L"` mode)
- Builds a `high_contrast` copy via `ImageEnhance.Contrast(...).enhance(2.0)`
- Thresholds with `gray.point(lambda x: 255 if x > 128 else 0)` — a hard cutoff at level 128 producing a binary image
- Prints the preprocessing note; the whole body is wrapped in `try/except: pass` so a missing file fails silently

**Watch:** the threshold is applied to the original `gray` image, not to the `high_contrast` copy — a common off-by-one in this chain. In practice, binarizing the *enhanced* image gives Tesseract the better input. Expected output: the single preprocessing line.

In [ ]:
from PIL import Image, ImageEnhance
try:
    img = Image.open("/tmp/test_resume.png")
    gray = img.convert("L")
    enhancer = ImageEnhance.Contrast(gray)
    high_contrast = enhancer.enhance(2.0)
    binary = gray.point(lambda x: 255 if x > 128 else 0)
    print("Preprocessing: grayscale + contrast + threshold = better OCR results")
except: pass

## Summary: OCR is a fallback for scanned PDFs. Preprocess images for best results.

**OCR is the last-resort extraction path — and the noisiest.** Only route to it when the file has no text layer; preprocess (grayscale → contrast → threshold) before running Tesseract; and treat the output as provisional, because character errors are guaranteed on real scans. It is also the path with the most moving parts (Python package + system binary), so error handling matters — Ch. 31 builds exactly that machinery.

Whatever branch produced the text — PDF (Ch. 26), DOCX (Ch. 27), or OCR — it now needs cleaning before any NLP runs. That cleaning is Ch. 29: text normalization for resumes.